In [1]:
import pandas as pd
import numpy as np

# =================CONFIGURATION =================
# 1. Path to your best submission file
SUBMISSION_PATH = "submission_eva02_adamw_384_tta_100ep_32batch.csv"#"submission_swin_384_tta_100ep_32batch.csv" 

# 2. Path to the original Test.csv 
TEST_CSV_PATH = "data/dataset/Unido_AfricaRice_Challenge/Test.csv" 

# 3. Output name
OUTPUT_PATH = "submission_eva02_adamw_384_tta_100ep_32batch_corrected.csv"
# ================================================

def optimize_submission():
    # Load data
    sub = pd.read_csv(SUBMISSION_PATH)
    test = pd.read_csv(TEST_CSV_PATH)
    
    # Check ID column consistency
    id_col = 'Image_ID' if 'Image_ID' in sub.columns else 'ID'
    
    # Merge 'Comment' column from Test.csv into Submission
    # We use 'left' merge to preserve the submission order
    sub = sub.merge(test[[id_col, 'Comment']], on=id_col, how='left')
    
    print("Applying Domain Rules...")
    
    # ---------------------------------------------------------
    # RULE 1: ROUNDING
    # All columns with 'Count' should be Integers
    # ---------------------------------------------------------
    count_cols = [c for c in sub.columns if 'Count' in c]
    for col in count_cols:
        # Round and convert to integer
        sub[col] = sub[col].apply(lambda x: max(0, x)) # Ensure no negatives
        sub[col] = sub[col].round().astype(int)
        
    # ---------------------------------------------------------
    # RULE 2: PADDY RICE LOGIC
    # Condition: Comment == 'Paddy'
    # ---------------------------------------------------------
    paddy_mask = sub['Comment'] == 'Paddy'
    print(f"Found {paddy_mask.sum()} rows labeled as 'Paddy'. Adjusting...")

    # A. Chalky_Count is always 0 for Paddy
    sub.loc[paddy_mask, 'Chalky_Count'] = 0
    
    # B. For Paddy, Count is the sum of Colors (Black, Red, Yellow, Green)
    # Note: We do this AFTER rounding so the sum is exact integers
    sub.loc[paddy_mask, 'Count'] = (
        sub.loc[paddy_mask, 'Black_Count'] + 
        sub.loc[paddy_mask, 'Red_Count'] + 
        sub.loc[paddy_mask, 'Yellow_Count'] + 
        sub.loc[paddy_mask, 'Green_Count']
    )

    # Clean up: Remove the helper 'Comment' column
    sub = sub.drop(columns=['Comment'])
    
    # Save
    sub.to_csv(OUTPUT_PATH, index=False)
    print(f"Optimization Complete! Saved to {OUTPUT_PATH}")


optimize_submission()

Applying Domain Rules...
Found 148 rows labeled as 'Paddy'. Adjusting...
Optimization Complete! Saved to submission_eva02_adamw_384_tta_100ep_32batch_corrected.csv
